# model-diff v0 — Llama 3.1 8B base vs Instruct

**Goal:** Compare two HF checkpoints (same architecture) at the weight level.

**Metrics per module:**
1. Frobenius norm of ΔW
2. Cosine similarity between W_old and W_new
3. SVD of ΔW → effective rank, top-k singular values, concentration ratio
4. Delta sparsity (fraction of params < threshold)

**Output:** JSON + static HTML report

- **Runtime:** CPU (0.07 units/hr) — all computation is CPU-only (no GPU needed)
- **Expected runtime:** ~15-30 min for 8B pair
- **Models:** meta-llama/Llama-3.1-8B vs meta-llama/Llama-3.1-8B-Instruct

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
!pip install -q safetensors huggingface_hub torch numpy matplotlib

from google.colab import drive, userdata
drive.mount('/content/drive')

import os
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN or ''

DRIVE_DIR = '/content/drive/MyDrive/model_diff'
for sub in ['results', 'reports']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

print('Drive root:', DRIVE_DIR)

In [ ]:
import gc
import json
import math
import os
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from huggingface_hub import snapshot_download, HfApi
from safetensors import safe_open


# ── Config ─────────────────────────────────────────────────────
CONFIG = {
    'model_a': 'meta-llama/Llama-3.1-8B',
    'model_b': 'meta-llama/Llama-3.1-8B-Instruct',
    'svd_top_k': 20,           # top singular values to report
    'sparsity_threshold': 1e-5, # ΔW elements below this are "zero"
    'min_params': 1000,         # skip tiny tensors (biases, norms)
    'results_path': f'{DRIVE_DIR}/results/v0_llama8b_base_vs_instruct.json',
    'report_path': f'{DRIVE_DIR}/reports/v0_llama8b_base_vs_instruct.html',
}

print('Config:', json.dumps(CONFIG, indent=2))

In [ ]:
# ── Download model shards (safetensors only, no full model load) ──

def download_safetensors(model_id):
    """Download only safetensors files + index. Returns local path."""
    path = snapshot_download(
        model_id,
        allow_patterns=['*.safetensors', '*.safetensors.index.json', 'config.json'],
        token=os.environ.get('HF_TOKEN'),
    )
    print(f'Downloaded {model_id} -> {path}')
    return path


def get_tensor_map(model_path):
    """Build dict: tensor_name -> shard_file for a sharded safetensors model."""
    index_file = os.path.join(model_path, 'model.safetensors.index.json')
    if os.path.exists(index_file):
        with open(index_file) as f:
            index = json.load(f)
        return {k: os.path.join(model_path, v) for k, v in index['weight_map'].items()}
    else:
        # Single-file model
        sf_file = os.path.join(model_path, 'model.safetensors')
        if not os.path.exists(sf_file):
            raise FileNotFoundError(f'No safetensors found in {model_path}')
        with safe_open(sf_file, framework='pt') as f:
            return {name: sf_file for name in f.keys()}


t0 = time.time()
path_a = download_safetensors(CONFIG['model_a'])
path_b = download_safetensors(CONFIG['model_b'])
print(f'Downloads done in {time.time()-t0:.0f}s')

tensor_map_a = get_tensor_map(path_a)
tensor_map_b = get_tensor_map(path_b)

# Verify same tensor names
names_a = set(tensor_map_a.keys())
names_b = set(tensor_map_b.keys())
if names_a != names_b:
    only_a = names_a - names_b
    only_b = names_b - names_a
    print(f'WARNING: tensor name mismatch!')
    if only_a: print(f'  Only in A: {sorted(only_a)[:10]}')
    if only_b: print(f'  Only in B: {sorted(only_b)[:10]}')
    common = sorted(names_a & names_b)
else:
    common = sorted(names_a)

print(f'Common tensors: {len(common)}')
print(f'Sample: {common[:5]}')

In [ ]:
# ── Core analysis: stream tensor-by-tensor ──────────────────────

def load_tensor(tensor_map, name):
    """Load a single tensor from safetensors shard."""
    shard_path = tensor_map[name]
    with safe_open(shard_path, framework='pt', device='cpu') as f:
        return f.get_tensor(name)


def effective_rank(sigma):
    """Shannon entropy-based effective rank from singular values."""
    s = sigma / sigma.sum()
    s = s[s > 1e-12]  # avoid log(0)
    return float(torch.exp(-(s * torch.log(s)).sum()))


def compute_cosine_sim(w_a_flat, w_b_flat):
    """Cosine similarity with clamp to [-1, 1]."""
    cos = torch.nn.functional.cosine_similarity(
        w_a_flat.unsqueeze(0), w_b_flat.unsqueeze(0)
    ).item()
    return float(max(-1.0, min(1.0, cos)))


def analyze_delta(w_a, w_b, top_k=20, sparsity_threshold=1e-5):
    """Compute all v0 metrics for a single weight delta."""
    shape = w_a.shape
    if w_a.ndim == 1:
        # 1D tensor (bias, norm) — no SVD, just norms
        delta = (w_b - w_a).float()
        return {
            'shape': list(shape),
            'n_params': int(w_a.numel()),
            'frob_norm': float(delta.norm()),
            'frob_norm_relative': float(delta.norm() / (w_a.float().norm() + 1e-12)),
            'cosine_sim': compute_cosine_sim(w_a.float().flatten(), w_b.float().flatten()),
            'sparsity': float((delta.abs() < sparsity_threshold).float().mean()),
            'has_svd': False,
        }

    # 2D+ tensor
    w_a_2d = w_a.float().reshape(shape[0], -1)
    w_b_2d = w_b.float().reshape(shape[0], -1)
    delta_2d = w_b_2d - w_a_2d

    frob = float(delta_2d.norm())
    frob_rel = frob / (float(w_a_2d.norm()) + 1e-12)
    cos_sim = compute_cosine_sim(w_a_2d.flatten(), w_b_2d.flatten())
    sparsity = float((delta_2d.abs() < sparsity_threshold).float().mean())

    # SVD (on CPU, float32)
    m, n = delta_2d.shape
    if min(m, n) > 8192:
        # Randomized SVD: project to lower dimension first
        k = min(top_k * 2, min(m, n))
        Q, _ = torch.linalg.qr(delta_2d @ torch.randn(n, k))
        B = Q.T @ delta_2d
        _, sigma, _ = torch.linalg.svd(B, full_matrices=False)
        sigma = sigma[:top_k]
    else:
        # Full SVD (only singular values needed)
        sigma = torch.linalg.svdvals(delta_2d)

    eff_rank = effective_rank(sigma)
    top_k_actual = min(top_k, len(sigma))
    top_sigmas = sigma[:top_k_actual]

    # Concentration: fraction of ||ΔW||_F captured by top-k SVs
    total_energy = float((sigma ** 2).sum())
    top_k_energy = float((top_sigmas ** 2).sum())
    concentration = top_k_energy / (total_energy + 1e-12)

    # Spectral decay: fit log(sigma) ~ -alpha * log(rank)
    ranks = torch.arange(1, len(sigma) + 1).float()
    log_sigma = torch.log(sigma[sigma > 1e-12])
    log_ranks = torch.log(ranks[:len(log_sigma)])
    if len(log_sigma) > 2:
        A = torch.stack([log_ranks, torch.ones_like(log_ranks)], dim=1)
        result = torch.linalg.lstsq(A, log_sigma)
        spectral_alpha = float(-result.solution[0])
    else:
        spectral_alpha = None

    return {
        'shape': list(shape),
        'n_params': int(w_a.numel()),
        'frob_norm': frob,
        'frob_norm_relative': frob_rel,
        'cosine_sim': cos_sim,
        'sparsity': sparsity,
        'has_svd': True,
        'effective_rank': eff_rank,
        'concentration_top_k': concentration,
        'top_k': top_k_actual,
        'top_singular_values': top_sigmas.tolist(),
        'spectral_alpha': spectral_alpha,
        'n_singular_values': int(len(sigma)),
    }


# ── Run analysis ──────────────────────────────────────────────
results = []
t0 = time.time()
skipped = 0

for i, name in enumerate(common):
    w_a = load_tensor(tensor_map_a, name)
    w_b = load_tensor(tensor_map_b, name)

    if w_a.numel() < CONFIG['min_params']:
        skipped += 1
        del w_a, w_b
        continue

    if w_a.shape != w_b.shape:
        print(f'  SKIP {name}: shape mismatch {w_a.shape} vs {w_b.shape}')
        skipped += 1
        del w_a, w_b
        continue

    metrics = analyze_delta(
        w_a, w_b,
        top_k=CONFIG['svd_top_k'],
        sparsity_threshold=CONFIG['sparsity_threshold'],
    )
    metrics['name'] = name
    results.append(metrics)

    del w_a, w_b
    gc.collect()

    if (i + 1) % 20 == 0 or (i + 1) == len(common):
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(common)}] {elapsed:.0f}s — last: {name}')

# Save checkpoint
with open(CONFIG['results_path'], 'w') as f:
    json.dump({
        'config': CONFIG,
        'model_a': CONFIG['model_a'],
        'model_b': CONFIG['model_b'],
        'n_tensors': len(results),
        'n_skipped': skipped,
        'modules': results,
    }, f, indent=2)

print(f'\nDone: {len(results)} tensors analyzed, {skipped} skipped, {time.time()-t0:.0f}s')
print(f'Saved to {CONFIG["results_path"]}')

In [ ]:
# ── Quick summary table ──────────────────────────────────────

with open(CONFIG['results_path']) as f:
    data = json.load(f)
modules = data['modules']

# Sort by relative Frobenius norm (most changed first)
by_frob = sorted(modules, key=lambda m: m['frob_norm_relative'], reverse=True)

print(f"model-diff: {data['model_a']} → {data['model_b']}")
print(f"{'Module':<55} {'ΔW/W':>8} {'cos_sim':>8} {'eff_rank':>9} {'conc_k':>7} {'sparsity':>8}")
print('─' * 100)
for m in by_frob[:30]:
    eff_r = f"{m['effective_rank']:.1f}" if m.get('has_svd') and m.get('effective_rank') else '—'
    conc = f"{m['concentration_top_k']:.3f}" if m.get('has_svd') and m.get('concentration_top_k') else '—'
    print(f"{m['name']:<55} {m['frob_norm_relative']:>8.4f} {m['cosine_sim']:>8.5f} {eff_r:>9} {conc:>7} {m['sparsity']:>8.3f}")

In [ ]:
# ── Visualization: layer-wise heatmap ─────────────────────────

import re

def parse_layer_info(name):
    """Extract layer number and module type from tensor name."""
    m = re.search(r'layers\.(\d+)\.(.+)', name)
    if m:
        return int(m.group(1)), m.group(2)
    return None, name


def short_module_name(mod_type):
    """Shorten module type for x-axis labels."""
    # self_attn.q_proj -> attn.q, mlp.gate_proj -> mlp.gate, etc.
    return (mod_type
        .replace('self_attn.', 'attn.')
        .replace('_proj', '')
        .replace('mlp.', 'mlp.')
        .replace('input_layernorm', 'ln_in')
        .replace('post_attention_layernorm', 'ln_post')
    )


# Build layer × module matrix
layer_modules = {}  # (layer_idx, module_type) -> metrics
module_types = set()
max_layer = 0

for m in modules:
    layer_idx, mod_type = parse_layer_info(m['name'])
    if layer_idx is not None:
        mod_type = mod_type.replace('.weight', '')
        layer_modules[(layer_idx, mod_type)] = m
        module_types.add(mod_type)
        max_layer = max(max_layer, layer_idx)

module_types = sorted(module_types)
n_layers = max_layer + 1

# Create matrices for plotting
frob_matrix = np.full((n_layers, len(module_types)), np.nan)
cos_matrix = np.full((n_layers, len(module_types)), np.nan)
rank_matrix = np.full((n_layers, len(module_types)), np.nan)
conc_matrix = np.full((n_layers, len(module_types)), np.nan)

for (li, mt), m in layer_modules.items():
    j = module_types.index(mt)
    frob_matrix[li, j] = m['frob_norm_relative']
    cos_matrix[li, j] = min(1.0, m['cosine_sim'])  # clamp for display
    if m.get('has_svd') and m.get('effective_rank'):
        rank_matrix[li, j] = m['effective_rank']
    if m.get('has_svd') and m.get('concentration_top_k'):
        conc_matrix[li, j] = m['concentration_top_k']

short_labels = [short_module_name(mt) for mt in module_types]

fig, axes = plt.subplots(2, 2, figsize=(14, 18))

plot_configs = [
    (axes[0, 0], frob_matrix, '||ΔW||/||W|| (relative Frobenius)', 'Reds', {}),
    (axes[0, 1], cos_matrix, 'Cosine similarity (W_old, W_new)', 'RdYlGn',
     {'vmin': np.nanmin(cos_matrix), 'vmax': 1.0}),
    (axes[1, 0], rank_matrix, 'Effective rank of ΔW', 'viridis', {}),
    (axes[1, 1], conc_matrix, f'Top-{CONFIG["svd_top_k"]} concentration', 'Oranges', {}),
]

for ax, matrix, title, cmap, extra_kw in plot_configs:
    im = ax.imshow(matrix, aspect='auto', cmap=cmap, interpolation='nearest', **extra_kw)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Module type', fontsize=9)
    ax.set_ylabel('Layer', fontsize=9)
    ax.set_xticks(range(len(module_types)))
    ax.set_xticklabels(short_labels, rotation=55, ha='right', fontsize=8)
    ax.set_yticks(range(0, n_layers, max(1, n_layers // 10)))
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(f"model-diff: {CONFIG['model_a']} → {CONFIG['model_b']}", fontsize=13, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f'{DRIVE_DIR}/reports/v0_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved heatmaps to {DRIVE_DIR}/reports/v0_heatmaps.png')

In [ ]:
# ── SVD spectral plots for top-changed modules ──────────────────

svd_modules = [m for m in modules if m.get('has_svd') and m.get('top_singular_values')]
top_changed = sorted(svd_modules, key=lambda m: m['frob_norm_relative'], reverse=True)[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, m in zip(axes.flat, top_changed):
    sigmas = np.array(m['top_singular_values'])
    ax.semilogy(range(1, len(sigmas) + 1), sigmas, 'b.-', markersize=4)
    ax.set_title(m['name'].split('model.')[-1], fontsize=9)
    ax.set_xlabel('rank')
    ax.set_ylabel('σ')
    info = f"eff_rank={m['effective_rank']:.1f}\nconc={m['concentration_top_k']:.3f}"
    if m.get('spectral_alpha'):
        info += f"\nα={m['spectral_alpha']:.2f}"
    ax.text(0.95, 0.95, info, transform=ax.transAxes, fontsize=7,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle(f"Top-9 changed modules: singular value spectra", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f'{DRIVE_DIR}/reports/v0_svd_spectra.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved SVD spectra to {DRIVE_DIR}/reports/v0_svd_spectra.png')

In [ ]:
# ── Generate static HTML report ─────────────────────────────────

import base64
from io import BytesIO


def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


# 1. Heatmaps (reuse short_labels and clamped cos_matrix from cell-6)
fig1, axes1 = plt.subplots(2, 2, figsize=(14, 18))
for ax, matrix, title, cmap, extra_kw in [
    (axes1[0, 0], frob_matrix, '||ΔW||/||W|| (relative Frobenius)', 'Reds', {}),
    (axes1[0, 1], cos_matrix, 'Cosine similarity (W_old, W_new)', 'RdYlGn',
     {'vmin': np.nanmin(cos_matrix), 'vmax': 1.0}),
    (axes1[1, 0], rank_matrix, 'Effective rank of ΔW', 'viridis', {}),
    (axes1[1, 1], conc_matrix, f'Top-{CONFIG["svd_top_k"]} concentration', 'Oranges', {}),
]:
    im = ax.imshow(matrix, aspect='auto', cmap=cmap, interpolation='nearest', **extra_kw)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Module type', fontsize=9)
    ax.set_ylabel('Layer', fontsize=9)
    ax.set_xticks(range(len(module_types)))
    ax.set_xticklabels(short_labels, rotation=55, ha='right', fontsize=8)
    ax.set_yticks(range(0, n_layers, max(1, n_layers // 10)))
    fig1.colorbar(im, ax=ax, shrink=0.8)
fig1.suptitle(f"model-diff: {CONFIG['model_a']} → {CONFIG['model_b']}", fontsize=13, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.97])
heatmap_b64 = fig_to_base64(fig1)
plt.close(fig1)

# 2. SVD spectra
fig2, axes2 = plt.subplots(3, 3, figsize=(15, 12))
for ax, m in zip(axes2.flat, top_changed):
    sigmas = np.array(m['top_singular_values'])
    ax.semilogy(range(1, len(sigmas) + 1), sigmas, 'b.-', markersize=4)
    ax.set_title(m['name'].split('model.')[-1], fontsize=9)
    ax.set_xlabel('rank')
    ax.set_ylabel('σ')
    info = f"eff_rank={m['effective_rank']:.1f}\nconc={m['concentration_top_k']:.3f}"
    if m.get('spectral_alpha'):
        info += f"\nα={m['spectral_alpha']:.2f}"
    ax.text(0.95, 0.95, info, transform=ax.transAxes, fontsize=7,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
fig2.suptitle(f"Top-9 changed modules: singular value spectra", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
svd_b64 = fig_to_base64(fig2)
plt.close(fig2)

# 3. Bar chart: per-layer total delta norm
layer_norms = np.zeros(n_layers)
for (li, mt), m in layer_modules.items():
    layer_norms[li] += m['frob_norm'] ** 2
layer_norms = np.sqrt(layer_norms)

fig3, ax3 = plt.subplots(figsize=(12, 4))
ax3.bar(range(n_layers), layer_norms, color='steelblue')
ax3.set_xlabel('Layer')
ax3.set_ylabel('||ΔW|| (Frobenius)')
ax3.set_title('Per-layer total weight delta magnitude')
plt.tight_layout()
layer_bar_b64 = fig_to_base64(fig3)
plt.close(fig3)

# Build summary stats
svd_mods = [m for m in modules if m.get('has_svd')]
mean_eff_rank = np.mean([m['effective_rank'] for m in svd_mods if m.get('effective_rank')])
mean_conc = np.mean([m['concentration_top_k'] for m in svd_mods if m.get('concentration_top_k')])
mean_cos = np.mean([min(1.0, m['cosine_sim']) for m in modules])
mean_sparsity = np.mean([m['sparsity'] for m in modules])
total_frob = math.sqrt(sum(m['frob_norm']**2 for m in modules))

# Build table rows
table_rows = ''
for m in by_frob:
    eff_r = f"{m['effective_rank']:.1f}" if m.get('has_svd') and m.get('effective_rank') else '—'
    conc = f"{m['concentration_top_k']:.3f}" if m.get('has_svd') and m.get('concentration_top_k') else '—'
    alpha = f"{m['spectral_alpha']:.2f}" if m.get('spectral_alpha') else '—'
    cos_display = min(1.0, m['cosine_sim'])
    table_rows += f"""<tr>
        <td style='font-family:monospace;font-size:12px'>{m['name']}</td>
        <td>{m['n_params']:,}</td>
        <td>{m['frob_norm_relative']:.4f}</td>
        <td>{cos_display:.5f}</td>
        <td>{eff_r}</td>
        <td>{conc}</td>
        <td>{m['sparsity']:.3f}</td>
        <td>{alpha}</td>
    </tr>\n"""

html = f"""<!DOCTYPE html>
<html><head>
<meta charset='utf-8'>
<title>model-diff: {CONFIG['model_a']} → {CONFIG['model_b']}</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 1400px; margin: 0 auto; padding: 20px; background: #fafafa; }}
  h1 {{ color: #333; border-bottom: 2px solid #4a90d9; padding-bottom: 10px; }}
  h2 {{ color: #555; margin-top: 30px; }}
  .summary {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin: 20px 0; }}
  .card {{ background: white; border-radius: 8px; padding: 15px; box-shadow: 0 1px 3px rgba(0,0,0,0.12); }}
  .card .value {{ font-size: 24px; font-weight: bold; color: #4a90d9; }}
  .card .label {{ font-size: 13px; color: #888; margin-top: 5px; }}
  img {{ max-width: 100%; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); margin: 10px 0; }}
  table {{ border-collapse: collapse; width: 100%; background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 3px rgba(0,0,0,0.12); }}
  th {{ background: #4a90d9; color: white; padding: 10px 12px; text-align: left; font-size: 13px; }}
  td {{ padding: 8px 12px; border-bottom: 1px solid #eee; font-size: 13px; }}
  tr:hover {{ background: #f5f8ff; }}
  .meta {{ color: #999; font-size: 12px; margin-top: 30px; }}
</style>
</head><body>

<h1>model-diff report</h1>
<p><strong>{CONFIG['model_a']}</strong> → <strong>{CONFIG['model_b']}</strong></p>

<div class='summary'>
  <div class='card'><div class='value'>{len(modules)}</div><div class='label'>Tensors analyzed</div></div>
  <div class='card'><div class='value'>{total_frob:.2f}</div><div class='label'>Total ||ΔW|| (Frobenius)</div></div>
  <div class='card'><div class='value'>{mean_cos:.5f}</div><div class='label'>Mean cosine similarity</div></div>
  <div class='card'><div class='value'>{mean_eff_rank:.1f}</div><div class='label'>Mean effective rank</div></div>
  <div class='card'><div class='value'>{mean_conc:.3f}</div><div class='label'>Mean top-{CONFIG['svd_top_k']} concentration</div></div>
  <div class='card'><div class='value'>{mean_sparsity:.3f}</div><div class='label'>Mean sparsity (|Δ|<1e-5)</div></div>
</div>

<h2>Per-layer delta magnitude</h2>
<img src='data:image/png;base64,{layer_bar_b64}' alt='Layer norms'>

<h2>Layer × Module heatmaps</h2>
<img src='data:image/png;base64,{heatmap_b64}' alt='Heatmaps'>

<h2>SVD spectra (top-9 changed modules)</h2>
<img src='data:image/png;base64,{svd_b64}' alt='SVD spectra'>

<h2>Full module table</h2>
<table>
<thead><tr>
  <th>Module</th><th>Params</th><th>||ΔW||/||W||</th><th>cos_sim</th><th>eff_rank</th><th>conc_top_k</th><th>sparsity</th><th>α (spectral)</th>
</tr></thead>
<tbody>
{table_rows}
</tbody>
</table>

<div class='meta'>
  Generated by model-diff v0 POC | SVD top_k={CONFIG['svd_top_k']} | sparsity_threshold={CONFIG['sparsity_threshold']}
</div>

</body></html>"""

with open(CONFIG['report_path'], 'w') as f:
    f.write(html)

print(f"HTML report saved to {CONFIG['report_path']}")
print(f"File size: {os.path.getsize(CONFIG['report_path']) / 1024:.0f} KB")

In [ ]:
# ── Summary ─────────────────────────────────────────────────────

print('=== model-diff v0 POC summary ===')
print(f"Models: {CONFIG['model_a']} → {CONFIG['model_b']}")
print(f"Tensors: {len(modules)} analyzed, {skipped} skipped")
print(f"Total ||ΔW||: {total_frob:.2f}")
print(f"Mean cosine similarity: {mean_cos:.5f}")
print(f"Mean effective rank: {mean_eff_rank:.1f}")
print(f"Mean top-{CONFIG['svd_top_k']} concentration: {mean_conc:.3f}")
print(f"Mean sparsity: {mean_sparsity:.3f}")
print()
print('Top-5 most changed modules:')
for m in by_frob[:5]:
    print(f"  {m['name']}: ΔW/W={m['frob_norm_relative']:.4f}, eff_rank={m.get('effective_rank', '—')}")
print()
print('Top-5 least changed modules:')
for m in by_frob[-5:]:
    print(f"  {m['name']}: ΔW/W={m['frob_norm_relative']:.6f}")
print()
print(f"Reports: {CONFIG['report_path']}")
print(f"Data: {CONFIG['results_path']}")